# Entra ID 3-Legged OAuth (3LO) for MCP Tool with URL Elicitation

This notebook demonstrates how to implement 3-Legged OAuth (Authorization Code flow) for an MCP tool that requires user consent to access protected resources using MCP's URL elicitation feature.

## Architecture

```
User ──[ID Token]──► Agent ──[M2M]──► MCP Server
                                         │
                                         │ Tool requires user consent
                                         ▼
                                    URL Elicitation
                                         │
                                         ▼
User ◄──────────────────────────── Auth URL returned
  │
  │ User consents at Entra ID
  ▼
AgentCore Token Vault ──► Protected API (Microsoft Graph)
```

## Key Concept: MCP URL Elicitation

MCP's URL elicitation (`elicit_url`) allows tools to request out-of-band user interactions:
- OAuth authorization flows
- Credential collection
- Any sensitive interaction that shouldn't pass through the LLM

When a tool needs user consent, it returns an authorization URL via elicitation. The client handles redirecting the user.

In [ ]:
!pip install -r requirements.txt -q

In [ ]:
import os
import uuid
import boto3
import urllib.parse
from boto3.session import Session
from bedrock_agentcore_starter_toolkit import Runtime

boto_session = Session()
sts = boto3.client('sts')
account_id = sts.get_caller_identity().get("Account")
region = boto_session.region_name or "us-west-2"

## Step 1: Configure Environment Variables

For 3LO, you need an additional app registration that will access Microsoft Graph on behalf of users.

In [ ]:
# Entra ID Configuration
os.environ["ENTRA_TENANT_ID"] = "your-tenant-id"  # Replace
os.environ["ENTRA_USER_APP_ID"] = "your-user-app-client-id"  # Replace
os.environ["ENTRA_MCP_APP_ID"] = "your-mcp-app-client-id"  # Replace
os.environ["ENTRA_AGENT_CLIENT_ID"] = "your-agent-client-id"  # Replace
os.environ["ENTRA_AGENT_CLIENT_SECRET"] = "your-agent-client-secret"  # Replace

# 3LO App - for accessing Microsoft Graph on behalf of users
os.environ["ENTRA_3LO_APP_ID"] = "your-3lo-app-client-id"  # Replace
os.environ["ENTRA_3LO_APP_SECRET"] = "your-3lo-app-secret"  # Replace

os.environ["ENTRA_USER_SCOPE"] = f"api://{os.environ['ENTRA_USER_APP_ID']}/.default openid profile"

## Step 2: Create 3LO Credential Provider

This credential provider handles the OAuth authorization code flow for user-delegated access.

In [ ]:
agentcore_client = boto3.client("bedrock-agentcore", region_name=region)

# Create 3LO credential provider
try:
    cp_response = agentcore_client.create_oauth2_credential_provider(
        name="entra-id-3lo-provider",
        credentialProviderVendor="MicrosoftEntraId",
        oauth2ProviderConfigInput={
            "microsoftEntraIdProviderConfig": {
                "tenantId": os.environ["ENTRA_TENANT_ID"],
                "clientId": os.environ["ENTRA_3LO_APP_ID"],
                "clientSecret": os.environ["ENTRA_3LO_APP_SECRET"]
            }
        }
    )
    callback_url = cp_response.get('callbackUrl', 'Check console for callback URL')
    print(f"Created 3LO credential provider")
    print(f"Add this callback URL to your Entra ID app: {callback_url}")
except agentcore_client.exceptions.ConflictException:
    print("3LO Credential provider already exists")

## Step 3: Create MCP Server with 3LO Tool using URL Elicitation

This MCP server includes a tool that uses URL elicitation to request user consent for Microsoft Graph access.

In [ ]:
%%writefile mcp_server_3lo.py
"""MCP Server with 3LO Tool using URL Elicitation"""
import os
import uuid
import httpx
from mcp.server.fastmcp import FastMCP
from mcp.server.elicitation import elicit_url, AcceptedUrlElicitation
from typing import Dict, Any

mcp = FastMCP(host="0.0.0.0", stateless_http=True)

TENANT_ID = os.environ.get("ENTRA_TENANT_ID")
CLIENT_ID = os.environ.get("ENTRA_3LO_APP_ID")
REDIRECT_URI = os.environ.get("OAUTH_CALLBACK_URL", "https://localhost/callback")

# In-memory token storage (use AgentCore Token Vault in production)
user_tokens = {}

def build_auth_url(user_id: str, scopes: list[str]) -> str:
    """Build Entra ID authorization URL."""
    state = f"{user_id}:{uuid.uuid4()}"
    scope = " ".join(scopes)
    params = {
        "client_id": CLIENT_ID,
        "response_type": "code",
        "redirect_uri": REDIRECT_URI,
        "scope": scope,
        "state": state,
        "response_mode": "query"
    }
    base_url = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/authorize"
    return f"{base_url}?{"&".join(f'{k}={v}' for k, v in params.items())}"

@mcp.tool()
def get_greeting(name: str) -> Dict[str, Any]:
    """Get a personalized greeting (no auth required)."""
    return {"greeting": f"Hello, {name}!"}

@mcp.tool()
async def get_user_profile() -> Dict[str, Any]:
    """
    Get the authenticated user's profile from Microsoft Graph.
    Requires 3LO - user must consent to allow access.
    """
    ctx = mcp.get_context()
    
    # Get user ID from request context (passed via headers)
    user_id = getattr(ctx, 'user_id', 'default_user')
    
    # Check if we have a token for this user
    user_token = user_tokens.get(user_id)
    
    if not user_token:
        # No token - use URL elicitation to request consent
        auth_url = build_auth_url(user_id, ["User.Read", "openid", "profile"])
        elicitation_id = f"auth-{user_id}-{uuid.uuid4()}"
        
        # Request URL elicitation from client
        result = await elicit_url(
            session=ctx.session,
            message="This tool requires access to your Microsoft profile. Please authorize access.",
            url=auth_url,
            elicitation_id=elicitation_id
        )
        
        if isinstance(result, AcceptedUrlElicitation):
            return {
                "status": "authorization_pending",
                "message": "User accepted authorization request. Waiting for callback.",
                "elicitation_id": elicitation_id
            }
        else:
            return {
                "status": "authorization_declined",
                "message": "User declined authorization request."
            }
    
    # We have a token - call Microsoft Graph
    try:
        async with httpx.AsyncClient() as client:
            response = await client.get(
                "https://graph.microsoft.com/v1.0/me",
                headers={"Authorization": f"Bearer {user_token}"}
            )
            response.raise_for_status()
            profile = response.json()
            
            return {
                "status": "success",
                "profile": {
                    "displayName": profile.get("displayName"),
                    "email": profile.get("mail") or profile.get("userPrincipalName"),
                    "jobTitle": profile.get("jobTitle")
                }
            }
    except Exception as e:
        return {"status": "error", "message": str(e)}

@mcp.tool()
def get_server_info() -> Dict[str, Any]:
    """Get information about this MCP server."""
    return {
        "server": "3LO Sample MCP Server",
        "tools": {
            "get_greeting": "No auth required",
            "get_user_profile": "Requires 3LO (User.Read) - uses URL elicitation"
        }
    }

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

## Step 4: Deploy MCP Server

In [ ]:
mcp_runtime = Runtime()

mcp_discovery_url = f"https://login.microsoftonline.com/{os.environ['ENTRA_TENANT_ID']}/.well-known/openid-configuration"
mcp_audience = f"api://{os.environ['ENTRA_MCP_APP_ID']}"

mcp_config = mcp_runtime.configure(
    entrypoint="mcp_server_3lo.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="entra_id_3lo_mcp",
    protocol_configuration={"serverProtocol": "MCP"},
    environment_variables={
        "ENTRA_TENANT_ID": os.environ["ENTRA_TENANT_ID"],
        "ENTRA_3LO_APP_ID": os.environ["ENTRA_3LO_APP_ID"]
    },
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": mcp_discovery_url,
            "allowedAudience": [mcp_audience]
        }
    }
)

mcp_launch = mcp_runtime.launch(local_build=True)
print(f"MCP Server deployed: {mcp_launch.agent_id}")

## Step 5: Create Agent with Elicitation Handling

In [ ]:
escaped_mcp_arn = urllib.parse.quote(mcp_launch.agent_arn, safe='')
mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_mcp_arn}/invocations?qualifier=DEFAULT"

In [ ]:
%%writefile agent_3lo.py
"""Agent with 3LO Elicitation Handling"""
import os
import boto3
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client
from mcp.types import ElicitResult
from bedrock_agentcore.runtime import BedrockAgentCoreApp

app = BedrockAgentCoreApp()

MCP_URL = os.environ.get("MCP_URL")
MCP_APP_ID = os.environ.get("ENTRA_MCP_APP_ID")

bedrock_model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
    temperature=0.1,
)

def get_m2m_token(workload_token: str) -> str:
    client = boto3.client("bedrock-agentcore")
    scope = f"api://{MCP_APP_ID}/.default"
    response = client.get_resource_oauth2_token(
        workloadIdentityToken=workload_token,
        resourceCredentialProviderName="entra-id-m2m-provider",
        scopes=[scope],
        oauth2Flow="M2M",
    )
    return response["accessToken"]

async def elicitation_callback(context, params):
    """Handle URL elicitation requests from MCP server."""
    print(f"ELICITATION REQUEST: {params.message}")
    print(f"Authorization URL: {params.url}")
    # In a real app, you'd redirect the user to this URL
    # For this sample, we accept and let the user handle it
    return ElicitResult(action="accept")

@app.entrypoint
def agent_handler(payload, context):
    prompt = payload.get("prompt", "hello")
    workload_token = context.get("workload_access_token")
    user_id = context.get("user_id", "default_user")
    
    m2m_token = get_m2m_token(workload_token)
    
    headers = {
        "authorization": f"Bearer {m2m_token}",
        "X-User-Id": user_id  # Pass user context to MCP
    }
    
    mcp_client = MCPClient(
        lambda: streamablehttp_client(MCP_URL, headers),
        elicitation_callback=elicitation_callback
    )
    
    with mcp_client:
        tools = mcp_client.list_tools_sync()
        agent = Agent(
            model=bedrock_model,
            tools=tools,
            system_prompt="""You are a helpful assistant. If a tool requires authorization,
            inform the user about the authorization URL they need to visit."""
        )
        response = agent(prompt)
    
    return str(response)

if __name__ == "__main__":
    app.run()

## Step 6: Deploy Agent

In [ ]:
agent_runtime = Runtime()

agent_discovery_url = f"https://login.microsoftonline.com/{os.environ['ENTRA_TENANT_ID']}/v2.0/.well-known/openid-configuration"

agent_config = agent_runtime.configure(
    entrypoint="agent_3lo.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="entra_id_3lo_agent",
    environment_variables={
        "MCP_URL": mcp_url,
        "ENTRA_MCP_APP_ID": os.environ["ENTRA_MCP_APP_ID"]
    },
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": agent_discovery_url,
            "allowedAudience": [os.environ["ENTRA_USER_APP_ID"]]
        }
    }
)

agent_launch = agent_runtime.launch(local_build=True)
print(f"Agent deployed: {agent_launch.agent_id}")

## Step 7: Test 3LO Flow

In [ ]:
import msal
import webbrowser

authority = f"https://login.microsoftonline.com/{os.environ['ENTRA_TENANT_ID']}"
scopes = [os.environ["ENTRA_USER_SCOPE"]]

msal_app = msal.PublicClientApplication(
    client_id=os.environ["ENTRA_USER_APP_ID"],
    authority=authority,
)

result = msal_app.acquire_token_silent(scopes, account=None)
if not result:
    flow = msal_app.initiate_device_flow(scopes=scopes)
    print(flow["message"])
    webbrowser.open(flow["verification_uri"])
    result = msal_app.acquire_token_by_device_flow(flow)

bearer_token = result["access_token"]
print(f"Bearer Token Received: {bearer_token[:30]}...")

In [ ]:
import requests
import json

escaped_agent_arn = urllib.parse.quote(agent_launch.agent_arn, safe='')
agent_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations?qualifier=DEFAULT"

session_id = str(uuid.uuid4())
headers = {
    "Authorization": f"Bearer {bearer_token}",
    "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id,
}

# This should trigger 3LO elicitation
response = requests.post(
    agent_url,
    data=json.dumps({"prompt": "Get my user profile from Microsoft Graph"}),
    headers=headers
)
print(response.text)

## Cleanup

In [ ]:
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=region)

agentcore_control.delete_agent_runtime(agentRuntimeId=agent_launch.agent_id)
agentcore_control.delete_agent_runtime(agentRuntimeId=mcp_launch.agent_id)
agentcore_client.delete_oauth2_credential_provider(name="entra-id-3lo-provider")
print("Cleanup complete")

## Conclusion

In this notebook we learned how to:
- Use MCP's URL elicitation feature for OAuth flows
- Create an MCP tool that requests user consent via elicitation
- Handle elicitation callbacks in the agent
- Implement 3-Legged OAuth with Entra ID for user-delegated access